In [0]:
# =============================================================================
# NOTEBOOK  : bronze_store_master
# PURPOSE   : Ingest store_master.csv → bronze.store_master
# LAYER     : Bronze (raw ingestion — no transformation)
# FREQUENCY : Weekly + incremental (Autoloader availableNow)
# FORMAT    : CSV
# =============================================================================
 
# ── 0. IMPORTS & CONFIG ──────────────────────────────────────────────────────
import sys, json
sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")
 
from utils.audit import start_audit, end_audit
from utils.quality_checks import assert_not_empty
from utils.schema_registry import BRONZE_STORE_MASTER_SCHEMA
 
from pyspark.sql.functions import current_timestamp, lit, col
 
with open("/Workspace/Shared/mini_project_a3t7/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_PATH  = cfg["landing_paths"]["store_master"]
TARGET_TABLE = cfg["tables"]["bronze_store_master"]
CHECKPOINT   = cfg["checkpoint_paths"]["bronze_store_master"]
PIPELINE     = "bronze_store_master"

In [0]:
# ── 1. START AUDIT ────────────────────────────────────────────────────────────
run_id = start_audit(spark, PIPELINE, TARGET_TABLE)
 
try:
    # ── 2. READ (Autoloader — availableNow) ───────────────────────────────────
    # availableNow = process all new files since last checkpoint, then stop
    # Autoloader tracks which files were already processed via checkpoint
    # On first run: processes all files in the directory
    # On subsequent runs: only processes newly added files
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", CHECKPOINT + "/schema")
        .schema(BRONZE_STORE_MASTER_SCHEMA)
        .load(SOURCE_PATH)
    )
 
    # ── 3. ADD AUDIT COLUMNS ──────────────────────────────────────────────────
    # Bronze rule: ONLY audit columns — zero business logic
    df = (
        df
        # .withColumn("source_file",     input_file_name())
        .withColumn("source_file",     col("_metadata.file_path"))
        .withColumn("ingested_at",     current_timestamp())
        .withColumn("pipeline_run_id", lit(run_id))
    )
 
    # ── 4. WRITE (streaming append + availableNow) ────────────────────────────
    # append — bronze keeps full raw history of every weekly incremental file
    # availableNow — behaves like batch, stops after processing new files
    query = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE)
    )
 
    query.awaitTermination()   # wait for batch to complete before proceeding
 
    # ── 5. QUALITY CHECK ──────────────────────────────────────────────────────
    # Read back what was just written to verify rows landed
    written_df = spark.read \
                      .table(TARGET_TABLE) \
                      .where(f"pipeline_run_id = '{run_id}'")
    rows_written = written_df.count()
    assert_not_empty(written_df, PIPELINE)
 
    print(f"[WRITE] {rows_written} rows written to {TARGET_TABLE}")
 
    # ── 6. END AUDIT (SUCCESS) ────────────────────────────────────────────────
    end_audit(spark, run_id, "SUCCESS", rows_written=rows_written)
 
except Exception as e:
    end_audit(spark, run_id, "FAILED", error=str(e))
    raise